In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = '/Users/Jiyeon/Desktop/ftp/fertility-treatment-prediction/data/raw'
train = pd.read_csv(f'{DATA_DIR}/train.csv')
test  = pd.read_csv(f'{DATA_DIR}/test.csv')

TARGET = '임신 성공 여부'
ID_COL = 'ID'

is_di_train = train['시술 유형'] == 'DI'
is_di_test  = test['시술 유형']  == 'DI'

print('Train:', train.shape, '/ Test:', test.shape)
print('DI 비율:', f"{is_di_train.mean()*100:.1f}%")

Train: (256351, 69) / Test: (90067, 68)
DI 비율: 2.5%


In [2]:
# 플래그 생성
# ── 기존 단계 진입 플래그 ──────────────────────────
FLAG_MAP = {
    '난자채취_진입여부': '난자 채취 경과일',
    '배아이식_진입여부': '배아 이식 경과일',
    '배아해동_진입여부': '배아 해동 경과일',
    '난자혼합_진입여부': '난자 혼합 경과일',
    'ICSI시술_진입여부': '미세주입된 난자 수',
    '배아저장_진입여부': '저장된 배아 수',
}
for flag, src in FLAG_MAP.items():
    if src in train.columns:
        train[flag] = train[src].notna().astype(int)
        test[flag]  = test[src].notna().astype(int)

# ── 시술 중단(Cycle Cancelled) 플래그 ─────────────────
# 난자는 채취됐는데 이식 배아 수가 0 → 배아 생성 실패 or 이식 취소
# IVF에서만 의미 있음 (DI는 배아 이식 자체 없음)
for df, is_di in [(train, is_di_train), (test, is_di_test)]:
    is_ivf = ~is_di
    egg_collected   = df['수집된 신선 난자 수'].fillna(0) > 0
    no_embryo_made  = df['총 생성 배아 수'].fillna(0) == 0
    no_transfer     = df['이식된 배아 수'].fillna(0) == 0

    # 배아 생성 실패: 난자는 있었는데 배아가 0
    df['배아생성_실패_플래그'] = (is_ivf & egg_collected & no_embryo_made).astype(int)

    # 이식 취소: 배아는 생겼는데 이식이 0
    embryo_made = df['총 생성 배아 수'].fillna(0) > 0
    df['이식취소_플래그'] = (is_ivf & embryo_made & no_transfer).astype(int)

    # 전체 사이클 중단 (IVF인데 난자 채취 자체 없음)
    df['사이클중단_플래그'] = (is_ivf & ~egg_collected & df['수집된 신선 난자 수'].notna()).astype(int)

print('플래그 생성 완료')
print('배아생성 실패:', train['배아생성_실패_플래그'].sum(), '건')
print('이식 취소    :', train['이식취소_플래그'].sum(), '건')
print('사이클 중단  :', train['사이클중단_플래그'].sum(), '건')

플래그 생성 완료
배아생성 실패: 10578 건
이식 취소    : 20245 건
사이클 중단  : 53845 건


In [3]:
# 경과일 컬럼 재처리
# ── 1-1. 배아 이식 경과일 → Day3 / Day5 / Day기타 / NA 범주형 ────
# 임상 기준: 3일차 = 세포기(Cleavage), 5일차 = 배반포(Blastocyst)

def classify_transfer_day(val):
    if pd.isna(val):
        return 'NA'        # 이식 단계 없음 (DI or 사이클 중단)
    # 간격 기반(배양_기간_일)이 있으면 그걸 우선, 없으면 절대값 활용
    # 여기서는 feature_engineering에서 만든 '배양_기간_일' 사용
    return None  # 아래에서 배양_기간_일 기반으로 처리

for df in [train, test]:
    if '배양_기간_일' in df.columns:
        df['이식_단계_범주'] = pd.cut(
            df['배양_기간_일'],
            bins=[-np.inf, 0, 3.5, 4.5, np.inf],
            labels=['NA_or_0일', 'Day3(세포기)', 'Day4(모룰라)', 'Day5이상(배반포)']
        ).astype(str)
        # NaN(이식 단계 없음)은 'NA'로
        df['이식_단계_범주'] = df['이식_단계_범주'].fillna('NA')
    else:
        # 배양_기간_일이 없으면 배아 이식 경과일 직접 사용
        df['이식_단계_범주'] = df['배아 이식 경과일'].apply(
            lambda x: 'NA' if pd.isna(x) else
                      'Day3(세포기)' if abs(x % 30 - 3) <= 0.5 else
                      'Day5이상(배반포)' if abs(x % 30 - 5) <= 1 else 'Day기타'
        )

print('이식 단계 범주 분포')
print(train['이식_단계_범주'].value_counts())

이식 단계 범주 분포
이식_단계_범주
Day5이상(배반포)    88736
Day기타          66125
Day3(세포기)      57924
NA             43566
Name: count, dtype: int64


In [4]:
# ── 1-2. 경과일 컬럼 DI 케이스 → -1 특이값 ───────────────────────
# NaN보다 -1이 명확: 트리가 '이 단계 없음'을 하나의 분기로 학습
# IVF 내 NaN은 그대로 (진짜 누락)

ELAPSED_COLS = [
    '난자 채취 경과일', '난자 해동 경과일', '난자 혼합 경과일',
    '배아 이식 경과일', '배아 해동 경과일'
]
ELAPSED_COLS = [c for c in ELAPSED_COLS if c in train.columns]

for col in ELAPSED_COLS:
    train.loc[is_di_train & train[col].isnull(), col] = -1
    test.loc[is_di_test   & test[col].isnull(),  col] = -1

# 간격 파생 피처도 DI는 -1
INTERVAL_COLS = ['배양_기간_일','채취_혼합_간격','채취_이식_총기간','해동_이식_간격']
INTERVAL_COLS = [c for c in INTERVAL_COLS if c in train.columns]

for col in INTERVAL_COLS:
    train.loc[is_di_train & train[col].isnull(), col] = -1
    test.loc[is_di_test   & test[col].isnull(),  col] = -1

print('경과일 DI 케이스 → -1 처리 완료')
print('IVF 내 경과일 NaN은 유지 (LGB/CatBoost 자체 처리)')

# 이식 단계 범주 vs 임신 성공률 확인
if TARGET in train.columns:
    print('\n=== 이식 단계별 임신 성공률 ===')
    display(
        train.groupby('이식_단계_범주')[TARGET]
             .agg(['mean','count'])
             .rename(columns={'mean':'성공률','count':'건수'})
             .sort_values('성공률', ascending=False)
             .style.format({'성공률':'{:.3f}'})
    )

경과일 DI 케이스 → -1 처리 완료
IVF 내 경과일 NaN은 유지 (LGB/CatBoost 자체 처리)

=== 이식 단계별 임신 성공률 ===


,성공률,건수
이식_단계_범주,,
Day5이상(배반포),0.398,88736
Day3(세포기),0.259,57924
Day기타,0.225,66125
NA,0.024,43566


In [5]:
# 배아 개수 DI -> 0 + IVF 잔존 결측처리
ZERO_IMPUTE_COLS = [
    '총 생성 배아 수','미세주입된 난자 수','미세주입에서 생성된 배아 수',
    '이식된 배아 수','미세주입 배아 이식 수','저장된 배아 수',
    '미세주입 후 저장된 배아 수','해동된 배아 수','해동 난자 수',
    '수집된 신선 난자 수','저장된 신선 난자 수','혼합된 난자 수',
    '파트너 정자와 혼합된 난자 수','기증자 정자와 혼합된 난자 수',
]
ZERO_IMPUTE_COLS = [c for c in ZERO_IMPUTE_COLS if c in train.columns]

# DI → 0
for col in ZERO_IMPUTE_COLS:
    train.loc[is_di_train & train[col].isnull(), col] = 0
    test.loc[is_di_test   & test[col].isnull(),  col] = 0

# IVF 잔존 결측 → NaN 유지 (LGB/CatBoost)
# XGBoost 사용 시 아래 주석 해제 (IVF 서브셋에서만 중앙값 fit)
# is_ivf_train = ~is_di_train
# is_ivf_test  = ~is_di_test
# for col in ZERO_IMPUTE_COLS:
#     med = train.loc[is_ivf_train, col].median()   # train IVF만 fit
#     train.loc[is_ivf_train & train[col].isnull(), col] = med
#     test.loc[is_ivf_test   & test[col].isnull(),  col] = med

print('배아 개수 DI→0 처리 완료')

배아 개수 DI→0 처리 완료


In [6]:
# 비율 피처 분모 0 재처리
def safe_ratio(numerator, denominator):
    """분모 0 → 0 반환 (1e-6 스무딩 없음)"""
    return np.where(denominator > 0, numerator / denominator, 0)

for df in [train, test]:
    # 비율 피처 재계산 (깨끗한 분모 처리)
    df['ICSI_수정률']    = safe_ratio(df['미세주입에서 생성된 배아 수'], df['미세주입된 난자 수'])
    df['배아_이식효율']  = safe_ratio(df['이식된 배아 수'],              df['총 생성 배아 수'])
    df['배아_저장비율']  = safe_ratio(df['저장된 배아 수'],              df['총 생성 배아 수'])
    df['전체_수정효율']  = safe_ratio(df['총 생성 배아 수'],             df['혼합된 난자 수'])
    df['난자_활용효율']  = safe_ratio(df['이식된 배아 수'],              df['수집된 신선 난자 수'])
    df['파트너정자_집중도'] = safe_ratio(df['파트너 정자와 혼합된 난자 수'], df['혼합된 난자 수'])

    # 해동 배아 비율 (분모: 총 생성 + 해동)
    denom = df['총 생성 배아 수'] + df['해동된 배아 수']
    df['해동배아_비율'] = safe_ratio(df['해동된 배아 수'], denom)

    # 진입불가 플래그: 분모가 0 = 해당 단계에 진입 못한 케이스
    df['난자수집_실패_플래그']  = (df['수집된 신선 난자 수'] == 0).astype(int)
    df['배아생성_제로_플래그']  = (df['총 생성 배아 수'] == 0).astype(int)
    df['난자혼합_제로_플래그']  = (df['혼합된 난자 수'] == 0).astype(int)

    # [0, 1] 클리핑 (혹시 모를 오차 제거)
    ratio_cols = [
        'ICSI_수정률','배아_이식효율','배아_저장비율','전체_수정효율',
        '해동배아_비율','난자_활용효율','파트너정자_집중도'
    ]
    for col in ratio_cols:
        df[col] = df[col].clip(0, 1)

print('비율 피처 재계산 완료 (1e-6 스무딩 제거)')
print('진입불가 플래그 생성 완료')
print()
print('난자수집 실패:', train['난자수집_실패_플래그'].sum(), '건')
print('배아생성 제로:', train['배아생성_제로_플래그'].sum(), '건')

비율 피처 재계산 완료 (1e-6 스무딩 제거)
진입불가 플래그 생성 완료

난자수집 실패: 60136 건
배아생성 제로: 59640 건


In [7]:
# 범주형 결측 처리 및 최종 검증
# 범주형 결측 → 'missing'
cat_cols = [
    c for c in train.columns
    if c not in [TARGET, ID_COL]
    and train[c].dtype == 'object'
]
for df in [train, test]:
    for col in cat_cols:
        df[col] = df[col].fillna('missing')
print(f'범주형 결측 처리: {len(cat_cols)}개 컬럼')

# Train/Test 정합성
only_train = set(train.columns) - {TARGET} - set(test.columns)
only_test  = set(test.columns) - set(train.columns)
if not only_train and not only_test:
    print('✅ Train/Test 컬럼 정합성 OK')
else:
    if only_train: print(f'⚠️ Train에만 있는 컬럼: {only_train}')
    if only_test:  print(f'⚠️ Test에만 있는 컬럼: {only_test}')

print(f'\n최종 Train: {train.shape} / Test: {test.shape}')

범주형 결측 처리: 0개 컬럼
✅ Train/Test 컬럼 정합성 OK

최종 Train: (256351, 89) / Test: (90067, 88)


In [8]:
# 새로 추가된 피처 전체 목록 출력
v3_new_features = [
    # 피드백 1
    '이식_단계_범주',
    # 피드백 2
    '배아생성_실패_플래그', '이식취소_플래그', '사이클중단_플래그',
    # 피드백 3
    '난자수집_실패_플래그', '배아생성_제로_플래그', '난자혼합_제로_플래그',
]
v3_new_features = [c for c in v3_new_features if c in train.columns]

print(f'v3에서 신규 추가 피처: {len(v3_new_features)}개')
display(train[v3_new_features].describe(include='all').T)

v3에서 신규 추가 피처: 7개


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
이식_단계_범주,256351,4,Day5이상(배반포),88736,NaN,NaN,NaN,NaN,NaN,NaN,NaN
배아생성_실패_플래그,256351.0,NaN,NaN,NaN,0.041264,0.1989,0.0,0.0,0.0,0.0,1.0
이식취소_플래그,256351.0,NaN,NaN,NaN,0.078974,0.269698,0.0,0.0,0.0,0.0,1.0
사이클중단_플래그,256351.0,NaN,NaN,NaN,0.210044,0.40734,0.0,0.0,0.0,0.0,1.0
난자수집_실패_플래그,256351.0,NaN,NaN,NaN,0.234585,0.42374,0.0,0.0,0.0,0.0,1.0
배아생성_제로_플래그,256351.0,NaN,NaN,NaN,0.23265,0.422522,0.0,0.0,0.0,0.0,1.0
난자혼합_제로_플래그,256351.0,NaN,NaN,NaN,0.206518,0.404807,0.0,0.0,0.0,0.0,1.0


In [9]:
train.to_csv('train_clean.csv', index=False, encoding='utf-8-sig')
test.to_csv('test_clean.csv',   index=False, encoding='utf-8-sig')
print('저장 완료: train_clean.csv / test_clean.csv')

저장 완료: train_clean.csv / test_clean.csv
